# NST FEVER Full Training — Colab T4 GPU

Complete FEVER fact verification experiments for the Neurosymbolic Transformers paper.

**Runtime:** ~3.5 hours on T4 | **Anti-leakage:** strict | **Reproducibility:** seeded

| Cell | Task | Time |
|------|------|------|
| 1 | Setup + GPU check | 2 min |
| 2 | Build wiki cache | 4 min |
| 3 | Smoke test | 2 min |
| 4 | Neural baseline (full) | ~45 min |
| 5 | NST Soft constraints | ~45 min |
| 6 | NST CEGIS | ~60 min |
| 7 | NST ECCG (Gated) | ~50 min |
| 8 | Results summary | instant |
| 9 | (Optional) NST Lagrangian | ~45 min |

> **Important:** Set runtime to **T4 GPU** before running: Runtime > Change runtime type > T4 GPU


In [ ]:
# Cell 1: Setup — Clone repo, install deps, verify GPU
import os, subprocess, sys, shutil, time

t0 = time.time()

# ── Check GPU ──
try:
    nvsmi = shutil.which("nvidia-smi")
    if nvsmi:
        r = subprocess.run([nvsmi, "--query-gpu=name,memory.total", "--format=csv,noheader"],
                           capture_output=True, text=True)
        gpu_name = r.stdout.strip()
        print(f"GPU: {gpu_name}")
        if "T4" not in gpu_name and "A100" not in gpu_name and "V100" not in gpu_name:
            print("WARNING: Not a T4/A100/V100. Training may be slow.")
    else:
        print("WARNING: No GPU detected! Go to Runtime > Change runtime type > T4 GPU")
except Exception as e:
    print(f"GPU check error: {e}")

# ── Clone repo ──
REPO = "nst"
if not os.path.exists(REPO):
    print("Cloning repo...")
    subprocess.run(["git", "clone", "https://github.com/poolanithinreddy/Neurosymbolic-Transformers.git", REPO], check=True)
    print("Repo cloned")
else:
    # Pull latest
    r = subprocess.run(["git", "pull", "--ff-only"], capture_output=True, text=True, cwd=REPO)
    print(f"git pull: {r.stdout.strip()}")

# ── cd into repo ──
os.chdir(REPO)
sys.path.insert(0, os.getcwd())
print(f"Working directory: {os.getcwd()}")

# ── Install dependencies ──
print("\nInstalling dependencies...")
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
    "torch", "transformers==4.46.3", "datasets==2.21.0",
    "sentencepiece>=0.1.99", "protobuf>=4.0",
    "pyyaml", "scikit-learn", "rank-bm25==0.2.2",
    "accelerate", "peft", "tiktoken",
    "fsspec>=2023.6,<2025", "huggingface_hub>=0.21,<1.0"],
    check=True, capture_output=True, text=True)
# Install repo in editable mode
subprocess.run([sys.executable, "-m", "pip", "install", "-e", ".", "--no-deps", "-q"],
    capture_output=True, text=True)

# ── Verify ──
import torch
import transformers
import datasets

print(f"\nPyTorch      : {torch.__version__}")
print(f"Transformers : {transformers.__version__}")
print(f"Datasets     : {datasets.__version__}")
print(f"CUDA         : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU          : {torch.cuda.get_device_name(0)}")
    print(f"VRAM         : {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
print(f"\nSetup complete ({time.time()-t0:.0f}s)")


In [ ]:
# Cell 2: Build FEVER wiki cache (one-time, ~4 min)
import os, sys, time

os.chdir("/content/nst")
sys.path.insert(0, "/content/nst")

# Clear any stale module refs
for mod in list(sys.modules.keys()):
    if any(mod.startswith(p) for p in ["data.", "models.", "training.", "eval.", "logic.", "symbolic.", "retrieval."]):
        del sys.modules[mod]

from data.fever_wiki_cache import WikiCache

cache_path = "data/fever_wiki.db"
if os.path.exists(cache_path):
    cache = WikiCache(cache_path)
    n = len(cache)
    print(f"Wiki cache already exists: {n} pages, {os.path.getsize(cache_path)/1024/1024:.1f} MB")
    cache.close()
else:
    print("Building wiki cache from HuggingFace FEVER dataset...")
    print("This downloads ~300MB and takes ~4 minutes. One-time only.")
    t0 = time.time()

    # Use the CLI command
    import subprocess
    r = subprocess.run([sys.executable, "main.py", "build-fever-wiki-cache"],
                       capture_output=True, text=True, timeout=600)
    print(r.stdout[-2000:] if r.stdout else "(no stdout)")
    if r.returncode != 0:
        print(f"STDERR: {r.stderr[-1000:]}")
    else:
        print(f"\nWiki cache built in {time.time()-t0:.0f}s")

# Verify
if os.path.exists(cache_path):
    cache = WikiCache(cache_path)
    n = len(cache)
    sample = cache.titles()[:3]
    for t in sample:
        sents = cache.lookup(t)
        print(f"  '{t}': {len(sents)} sentences")
    cache.close()
    print(f"Wiki cache verified: {n} pages")
else:
    print("ERROR: Wiki cache not found! Check errors above.")


In [ ]:
# Cell 3: Smoke Test — DeBERTa on 200 examples (~2 min on T4)
import os, sys, time, logging

os.chdir("/content/nst")
sys.path.insert(0, "/content/nst")

# Clear stale modules
for mod in list(sys.modules.keys()):
    if any(mod.startswith(p) for p in ["data.", "models.", "training.", "eval.", "logic.", "symbolic.", "retrieval."]):
        del sys.modules[mod]

logging.basicConfig(level=logging.INFO, format="%(name)s | %(message)s", force=True)

print("=" * 60)
print("  SMOKE TEST: DeBERTa-v3-base, 200 train, 100 dev, 1 epoch")
print("  Purpose: Validate full pipeline before long runs")
print("=" * 60 + "\n")

t0 = time.time()
from training.train_fever_nst import train_fever_nst
results = train_fever_nst("configs/fever_gold_smoke.yaml")
elapsed = time.time() - t0

# Show results
print("\n" + "=" * 60)
print(f"SMOKE TEST RESULTS ({elapsed:.0f}s):")
print("=" * 60)
if isinstance(results, dict):
    dev = results.get("dev", {})
    print(f"  dev_accuracy : {dev.get('accuracy', 'N/A')}")
    print(f"  dev_ece      : {dev.get('ece', 'N/A')}")
    print(f"  dev_brier    : {dev.get('brier', 'N/A')}")
    print(f"  temperature  : {results.get('temperature', 'N/A')}")
    print(f"  nan_abort    : {results.get('nan_abort', False)}")

    if results.get("nan_abort", False):
        print("\nSMOKE TEST FAILED -- NaN detected!")
    else:
        print("\nSmoke test passed! Pipeline is working correctly.")
else:
    print(f"  Unexpected result type: {type(results)}")


In [ ]:
# Cell 4: NEURAL BASELINE — Full data, 3 epochs (~45 min on T4)
# Setting A: Gold Evidence, pure cross-entropy, NO constraints.
import os, sys, time, logging, json

os.chdir("/content/nst")
sys.path.insert(0, "/content/nst")

# Clear stale modules
for mod in list(sys.modules.keys()):
    if any(mod.startswith(p) for p in ["data.", "models.", "training.", "eval.", "logic.", "symbolic.", "retrieval."]):
        del sys.modules[mod]

logging.basicConfig(level=logging.INFO, format="%(name)s | %(message)s", force=True)

print("=" * 60)
print("  NEURAL BASELINE (Setting A: Gold Evidence)")
print("  Model: DeBERTa-v3-base (184M params)")
print("  Data: Full FEVER train (~145K) + dev (~19K)")
print("  Epochs: 3, batch=16, grad_accum=2 (eff. batch=32)")
print("  Eval: every 500 steps on 2K dev subset")
print("  No constraints (lambda=0)")
print("=" * 60 + "\n")

t0 = time.time()
from training.train_fever_nst import train_fever_nst
results_neural = train_fever_nst("configs/fever_gold_neural.yaml")
elapsed = time.time() - t0

# Show results
print("\n" + "=" * 60)
print(f"NEURAL BASELINE RESULTS ({elapsed/60:.1f} min):")
print("=" * 60)
dev = results_neural.get("dev", {})
dev_test = results_neural.get("dev_test", {})
print(f"  dev accuracy      : {dev.get('accuracy', 'N/A')}")
print(f"  dev ECE           : {dev.get('ece', 'N/A')}")
print(f"  dev Brier         : {dev.get('brier', 'N/A')}")
if dev_test:
    print(f"  dev_test accuracy : {dev_test.get('accuracy', 'N/A')}")
    print(f"  dev_test ECE      : {dev_test.get('ece', 'N/A')}")
print(f"  temperature       : {results_neural.get('temperature', 'N/A')}")
print(f"  best_dev_acc      : {results_neural.get('best_dev_acc', 'N/A')}")
print(f"\nPer-label:")
for label, stats in dev.get("per_label", {}).items():
    print(f"    {label}: acc={stats.get('accuracy', 0):.4f} (n={stats.get('count', 0)})")

# Save for comparison
with open("results_neural.json", "w") as f:
    json.dump(results_neural, f, indent=2)
print(f"\nNeural baseline done in {elapsed/60:.1f} min")
print(f"   Checkpoint: outputs_fever_gold_neural/ckpt/")


In [ ]:
# Cell 5: NST SOFT CONSTRAINTS — Fixed lambda=0.1 (~45 min on T4)
# Cross-entropy + lambda * constraint_loss (fixed weight)
import os, sys, time, logging, json

os.chdir("/content/nst")
sys.path.insert(0, "/content/nst")

for mod in list(sys.modules.keys()):
    if any(mod.startswith(p) for p in ["data.", "models.", "training.", "eval.", "logic.", "symbolic.", "retrieval."]):
        del sys.modules[mod]

logging.basicConfig(level=logging.INFO, format="%(name)s | %(message)s", force=True)

print("=" * 60)
print("  NST SOFT CONSTRAINTS (Setting A: Gold Evidence)")
print("  CE + fixed lambda=0.1 x constraint_loss")
print("  5 constraints: date, number, negation, entity, empty")
print("=" * 60 + "\n")

t0 = time.time()
from training.train_fever_nst import train_fever_nst
results_soft = train_fever_nst("configs/fever_gold_nst_soft.yaml")
elapsed = time.time() - t0

dev = results_soft.get("dev", {})
dev_test = results_soft.get("dev_test", {})
print("\n" + "=" * 60)
print(f"NST SOFT RESULTS ({elapsed/60:.1f} min):")
print("=" * 60)
print(f"  dev accuracy      : {dev.get('accuracy', 'N/A')}")
print(f"  dev ECE           : {dev.get('ece', 'N/A')}")
if dev_test:
    print(f"  dev_test accuracy : {dev_test.get('accuracy', 'N/A')}")
print(f"  final lambda      : {results_soft.get('final_lambda', 'N/A')}")

with open("results_soft.json", "w") as f:
    json.dump(results_soft, f, indent=2)
print(f"\nNST Soft done in {elapsed/60:.1f} min")


In [ ]:
# Cell 6: NST CEGIS — Counterexample-Guided (~60 min on T4)
# Lagrangian + counterexample mining from TRAINING SET (no leakage)
import os, sys, time, logging, json

os.chdir("/content/nst")
sys.path.insert(0, "/content/nst")

for mod in list(sys.modules.keys()):
    if any(mod.startswith(p) for p in ["data.", "models.", "training.", "eval.", "logic.", "symbolic.", "retrieval."]):
        del sys.modules[mod]

logging.basicConfig(level=logging.INFO, format="%(name)s | %(message)s", force=True)

print("=" * 60)
print("  NST CEGIS (Setting A: Gold Evidence)")
print("  Lagrangian + counterexample-guided outer loop")
print("  Max 5 CEGIS rounds, counterexamples mined from TRAIN only")
print("=" * 60 + "\n")

t0 = time.time()
from training.train_fever_nst import train_fever_nst
results_cegis = train_fever_nst("configs/fever_gold_nst_cegis.yaml")
elapsed = time.time() - t0

dev = results_cegis.get("dev", {})
dev_test = results_cegis.get("dev_test", {})
cegis_info = results_cegis.get("cegis", {})
print("\n" + "=" * 60)
print(f"NST CEGIS RESULTS ({elapsed/60:.1f} min):")
print("=" * 60)
print(f"  dev accuracy      : {dev.get('accuracy', 'N/A')}")
print(f"  dev ECE           : {dev.get('ece', 'N/A')}")
if dev_test:
    print(f"  dev_test accuracy : {dev_test.get('accuracy', 'N/A')}")
print(f"  CEGIS rounds      : {cegis_info.get('total_rounds', 'N/A')}")
print(f"  CEGIS converged   : {cegis_info.get('converged', 'N/A')}")
print(f"  final lambda      : {results_cegis.get('final_lambda', 'N/A')}")

with open("results_cegis.json", "w") as f:
    json.dump(results_cegis, f, indent=2)
print(f"\nNST CEGIS done in {elapsed/60:.1f} min")


In [ ]:
# Cell 7: NST ECCG (GATED) — Evidence-Conditioned Constraint Gating (~50 min)
# Novel contribution: learned per-sample, per-constraint gates
import os, sys, time, logging, json

os.chdir("/content/nst")
sys.path.insert(0, "/content/nst")

for mod in list(sys.modules.keys()):
    if any(mod.startswith(p) for p in ["data.", "models.", "training.", "eval.", "logic.", "symbolic.", "retrieval."]):
        del sys.modules[mod]

logging.basicConfig(level=logging.INFO, format="%(name)s | %(message)s", force=True)

print("=" * 60)
print("  NST ECCG / GATED (Setting A: Gold Evidence)")
print("  Evidence-Conditioned Constraint Gating (NOVEL)")
print("  Learns when to apply each constraint per-sample")
print("=" * 60 + "\n")

t0 = time.time()
from training.train_fever_nst import train_fever_nst
results_gated = train_fever_nst("configs/fever_gold_nst_gated.yaml")
elapsed = time.time() - t0

dev = results_gated.get("dev", {})
dev_test = results_gated.get("dev_test", {})
print("\n" + "=" * 60)
print(f"NST ECCG RESULTS ({elapsed/60:.1f} min):")
print("=" * 60)
print(f"  dev accuracy      : {dev.get('accuracy', 'N/A')}")
print(f"  dev ECE           : {dev.get('ece', 'N/A')}")
if dev_test:
    print(f"  dev_test accuracy : {dev_test.get('accuracy', 'N/A')}")
print(f"  final lambda      : {results_gated.get('final_lambda', 'N/A')}")

with open("results_gated.json", "w") as f:
    json.dump(results_gated, f, indent=2)
print(f"\nNST ECCG done in {elapsed/60:.1f} min")


In [ ]:
# Cell 8: Results Summary — Comparison Table
import json, os

os.chdir("/content/nst")

print("=" * 70)
print("  FEVER RESULTS -- Setting A: Gold Evidence, DeBERTa-v3-base")
print("=" * 70)
print()

# Load all results
results = {}
for name, path in [("Neural", "results_neural.json"),
                    ("Soft", "results_soft.json"),
                    ("CEGIS", "results_cegis.json"),
                    ("ECCG", "results_gated.json")]:
    if os.path.exists(path):
        with open(path) as f:
            results[name] = json.load(f)
    else:
        print(f"  {name}: not found ({path})")

# Try loading optional lagrangian
if os.path.exists("results_lagrangian.json"):
    with open("results_lagrangian.json") as f:
        results["Lagrangian"] = json.load(f)

if not results:
    print("No results found! Run cells 4-7 first.")
else:
    # Print table header
    header = f"  {'Mode':<12} {'Dev Acc':<10} {'ECE down':<10} {'Brier down':<10} {'DevTest Acc':<12} {'lambda':<8}"
    print(header)
    print("  " + "-" * 68)

    for name, r in results.items():
        dev = r.get("dev", {})
        dt = r.get("dev_test", {})
        lam = r.get("final_lambda", 0)
        dev_acc = dev.get("accuracy", 0)
        dev_ece = dev.get("ece", 0)
        dev_brier = dev.get("brier", 0)
        dt_acc = dt.get("accuracy", 0) if dt else "N/A"
        if isinstance(dt_acc, float):
            dt_str = f"{dt_acc:<12.4f}"
        else:
            dt_str = f"{dt_acc:<12}"
        print(f"  {name:<12} {dev_acc:<10.4f} {dev_ece:<10.4f} {dev_brier:<10.4f} {dt_str} {lam:<8.4f}")

    print()
    print("  Key:")
    print("  - Dev Acc: Label accuracy on dev set (tuning split)")
    print("  - DevTest Acc: Label accuracy on held-out 10% (final number)")
    print("  - ECE: Expected Calibration Error (lower = better calibrated)")
    print("  - Brier: Brier score (lower = better)")
    print("  - lambda: Final Lagrangian multiplier value")

    # Find best
    best_name = max(results, key=lambda n: results[n].get("dev", {}).get("accuracy", 0))
    best_acc = results[best_name].get("dev", {}).get("accuracy", 0)
    print(f"\n  Best: {best_name} (dev_acc={best_acc:.4f})")

    # Per-label breakdown for best model
    best_dev = results[best_name].get("dev", {})
    print(f"\n  Per-label breakdown ({best_name}):")
    for label, stats in best_dev.get("per_label", {}).items():
        print(f"    {label}: acc={stats.get('accuracy', 0):.4f} (n={stats.get('count', 0)})")

print("\nAll experiments complete!")
print("Full reports saved in outputs_fever_gold_*/report.json")
print("\nRemember to stop the Colab runtime to save GPU hours!")
print("    Runtime > Disconnect and delete runtime")


In [ ]:
# Cell 9 (OPTIONAL): NST LAGRANGIAN — Adaptive lambda (~45 min on T4)
# Run this if you want the full 5-mode ablation table
import os, sys, time, logging, json

os.chdir("/content/nst")
sys.path.insert(0, "/content/nst")

for mod in list(sys.modules.keys()):
    if any(mod.startswith(p) for p in ["data.", "models.", "training.", "eval.", "logic.", "symbolic.", "retrieval."]):
        del sys.modules[mod]

logging.basicConfig(level=logging.INFO, format="%(name)s | %(message)s", force=True)

print("=" * 60)
print("  NST LAGRANGIAN (Setting A: Gold Evidence)")
print("  CE + adaptive Lagrangian constraint weighting")
print("=" * 60 + "\n")

t0 = time.time()
from training.train_fever_nst import train_fever_nst
results_lag = train_fever_nst("configs/fever_gold_lagrangian.yaml")
elapsed = time.time() - t0

dev = results_lag.get("dev", {})
dev_test = results_lag.get("dev_test", {})
print("\n" + "=" * 60)
print(f"NST LAGRANGIAN RESULTS ({elapsed/60:.1f} min):")
print("=" * 60)
print(f"  dev accuracy      : {dev.get('accuracy', 'N/A')}")
print(f"  dev ECE           : {dev.get('ece', 'N/A')}")
if dev_test:
    print(f"  dev_test accuracy : {dev_test.get('accuracy', 'N/A')}")
print(f"  final lambda      : {results_lag.get('final_lambda', 'N/A')}")

with open("results_lagrangian.json", "w") as f:
    json.dump(results_lag, f, indent=2)
print(f"\nNST Lagrangian done in {elapsed/60:.1f} min")
